In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
import os

class DTICNN(nn.Module):
    def __init__(self, ligand_dim=1024, protein_dim=320, conv_channels=[64, 128], 
                 fc_dims=[512, 256, 1], dropout_rate=0.3):
        super(DTICNN, self).__init__()
        
        # Ligand processing branch - ensure 3D input for conv1d
        self.ligand_conv = nn.Sequential(
            nn.Conv1d(1, conv_channels[0], kernel_size=5, padding=2),  # [batch, 1, 1024] -> [batch, 64, 1024]
            nn.BatchNorm1d(conv_channels[0]),
            nn.ReLU(),
            nn.MaxPool1d(2),  # [batch, 64, 1024] -> [batch, 64, 512]
            
            nn.Conv1d(conv_channels[0], conv_channels[1], kernel_size=3, padding=1),  # [batch, 64, 512] -> [batch, 128, 512]
            nn.BatchNorm1d(conv_channels[1]),
            nn.ReLU(),
            nn.MaxPool1d(2),  # [batch, 128, 512] -> [batch, 128, 256]
        )
        
        # Protein processing - using fully connected layers since 320 is small
        self.protein_fc = nn.Sequential(
            nn.Linear(protein_dim, conv_channels[0]),
            nn.BatchNorm1d(conv_channels[0]),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(conv_channels[0], conv_channels[1]),
            nn.BatchNorm1d(conv_channels[1]),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        
        # Calculate flattened dimensions
        ligand_flat_dim = conv_channels[1] * 256  # 128 * 256 = 32768
        protein_flat_dim = conv_channels[1]       # 128
        
        # Combined fully connected layers
        self.fc_layers = nn.Sequential(
            nn.Linear(ligand_flat_dim + protein_flat_dim, fc_dims[0]),
            nn.BatchNorm1d(fc_dims[0]),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(fc_dims[0], fc_dims[1]),
            nn.BatchNorm1d(fc_dims[1]),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(fc_dims[1], fc_dims[2])
        )
        
    def forward(self, ligand_features, protein_features):
        # CRITICAL FIX: Ensure ligand_features is exactly 2D [batch, 1024]
        if ligand_features.dim() == 4:
            # If it's [batch, 1, 1, 1024], squeeze the extra dimensions
            ligand_features = ligand_features.squeeze(1).squeeze(1)  # Remove both extra 1s
        elif ligand_features.dim() == 3 and ligand_features.size(1) == 1:
            # If it's [batch, 1, 1024], squeeze the middle dimension
            ligand_features = ligand_features.squeeze(1)
        elif ligand_features.dim() != 2:
            # If it's anything else unexpected, reshape to [batch, 1024]
            ligand_features = ligand_features.view(ligand_features.size(0), -1)
        
        # Now ligand_features should be [batch, 1024]
        # Add channel dimension for conv1d: [batch, 1, 1024]
        lig_x = ligand_features.unsqueeze(1)
        
        # Apply convolutions: [batch, 1, 1024] -> [batch, 128, 256]
        lig_x = self.ligand_conv(lig_x)
        # Flatten: [batch, 128, 256] -> [batch, 32768]
        lig_x = lig_x.view(lig_x.size(0), -1)
        
        # Process protein: [batch, 320] -> [batch, 128]
        prot_x = self.protein_fc(protein_features)
        
        # Concatenate: [batch, 32768 + 128] = [batch, 32896]
        combined = torch.cat((lig_x, prot_x), dim=1)
        
        # Final prediction
        output = self.fc_layers(combined)
        return output

def load_preprocessed_data(data_dir='data/processed_features'):
    """Load preprocessed ligand and protein features"""
    X_lig = np.load(os.path.join(data_dir, 'X_lig.npy')).astype(np.float32)
    X_prot = np.load(os.path.join(data_dir, 'X_prot.npy')).astype(np.float32)
    y = np.load(os.path.join(data_dir, 'y.npy')).astype(np.float32)
    
    print(f"Ligand features shape: {X_lig.shape}")  # Should be [N, 1024]
    print(f"Protein features shape: {X_prot.shape}")  # Should be [N, 320]
    print(f"Target values shape: {y.shape}")  # Should be [N,] or [N, 1]
    
    return X_lig, X_prot, y

def train_model(model, train_loader, val_loader, num_epochs=15, lr=0.01):
    """Train the CNN model"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    
    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0
        for batch_idx, (ligand_batch, protein_batch, target_batch) in enumerate(train_loader):
            # CRITICAL: Ensure proper tensor shapes before moving to device
            if ligand_batch.dim() == 4:
                ligand_batch = ligand_batch.squeeze(1).squeeze(1)  # Remove extra dims
            elif ligand_batch.dim() == 3 and ligand_batch.size(1) == 1:
                ligand_batch = ligand_batch.squeeze(1)  # Remove channel dim
            
            # Verify shapes before proceeding
            assert ligand_batch.dim() == 2, f"Ligand batch should be 2D, got {ligand_batch.shape}"
            assert ligand_batch.size(1) == 1024, f"Ligand batch should have 1024 features, got {ligand_batch.size(1)}"
            
            ligand_batch = ligand_batch.to(device)
            protein_batch = protein_batch.to(device)
            target_batch = target_batch.to(device).unsqueeze(1)
            
            optimizer.zero_grad()
            outputs = model(ligand_batch, protein_batch)
            loss = criterion(outputs, target_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        # Validation phase
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for ligand_batch, protein_batch, target_batch in val_loader:
                # Same shape checking for validation data
                if ligand_batch.dim() == 4:
                    ligand_batch = ligand_batch.squeeze(1).squeeze(1)
                elif ligand_batch.dim() == 3 and ligand_batch.size(1) == 1:
                    ligand_batch = ligand_batch.squeeze(1)
                
                ligand_batch = ligand_batch.to(device)
                protein_batch = protein_batch.to(device)
                target_batch = target_batch.to(device).unsqueeze(1)
                
                outputs = model(ligand_batch, protein_batch)
                val_loss += criterion(outputs, target_batch).item()
        
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        scheduler.step(avg_val_loss)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_dti_cnn.pth')
        
        # if epoch % 10 == 0:
        print(f'Epoch {epoch}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')
    
    return model

def evaluate_model(model, test_loader):
    """Evaluate model performance"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for ligand_batch, protein_batch, target_batch in test_loader:
            # Handle tensor shapes for evaluation
            if ligand_batch.dim() == 4:
                ligand_batch = ligand_batch.squeeze(1).squeeze(1)
            elif ligand_batch.dim() == 3 and ligand_batch.size(1) == 1:
                ligand_batch = ligand_batch.squeeze(1)
            
            ligand_batch = ligand_batch.to(device)
            protein_batch = protein_batch.to(device)
            target_batch = target_batch.to(device)
            
            outputs = model(ligand_batch, protein_batch)
            all_preds.extend(outputs.cpu().numpy())
            all_targets.extend(target_batch.cpu().numpy())
    
    mse = mean_squared_error(all_targets, all_preds)
    r2 = r2_score(all_targets, all_preds)
    print(f'Test MSE: {mse:.4f}, R²: {r2:.4f}')

def main():
    # Load data
    X_lig, X_prot, y = load_preprocessed_data()
    
    # Normalize features
    X_lig = (X_lig - X_lig.mean(axis=0)) / (X_lig.std(axis=0) + 1e-8)
    X_prot = (X_prot - X_prot.mean(axis=0)) / (X_prot.std(axis=0) + 1e-8)
    
    # Create datasets
    dataset = TensorDataset(
        torch.tensor(X_lig),      # [N, 1024] - should remain 2D
        torch.tensor(X_prot),     # [N, 320] - should remain 2D
        torch.tensor(y)           # [N,] or [N, 1]
    )
    
    # Split data
    train_size = int(0.8 * len(dataset))
    val_size = int(0.1 * len(dataset))
    test_size = len(dataset) - train_size - val_size
    
    train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
        dataset, [train_size, val_size, test_size]
    )
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    # Initialize model
    model = DTICNN(
        ligand_dim=1024,
        protein_dim=320,
        conv_channels=[64, 128],
        fc_dims=[512, 256, 1],
        dropout_rate=0.3
    )
    
    print(f"Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters")
    
    # Train model
    trained_model = train_model(model, train_loader, val_loader, num_epochs=100)
    
    # Evaluate
    evaluate_model(trained_model, test_loader)

if __name__ == "__main__":
    main()

Ligand features shape: (9867, 1024)
Protein features shape: (9867, 320)
Target values shape: (9867,)
Model initialized with 17,031,105 parameters


KeyboardInterrupt: 